In [10]:
import pandas as pd
import re

import torch

In [11]:
#extracting datasets
df_cnbc = pd.read_csv('Dataset/cnbc_headlines.csv')
df_guardian = pd.read_csv('Dataset/guardian_headlines.csv')
df_reuters = pd.read_csv('Dataset/reuters_headlines.csv')

#considering only headlines
headline_cnbc = list(df_cnbc['Headlines'])
headline_guardian = list(df_guardian['Headlines'])
headline_reuters = list(df_reuters['Headlines'])

#list of headlines
headline_list = headline_cnbc + headline_guardian + headline_reuters

print("No.of headlines in list: ",len(headline_list))
print(headline_list[:5])


No.of headlines in list:  53650
['Jim Cramer: A better way to invest in the Covid-19 vaccine gold rush', "Cramer's lightning round: I would own Teradyne", nan, "Cramer's week ahead: Big week for earnings, even bigger week for vaccines", 'IQ Capital CEO Keith Bliss says tech and healthcare will rally']


In [12]:
#cleaning dataset

#filtering empty rows
headline_list = [h for h in headline_list if str(h) != 'nan']
print("No.of headlines in list: ",len(headline_list))
print(headline_list[:5])

#removing spaces in headline
def clean_headline(h):
    h = h.strip() #remove the trailing spaces from either end
    h = re.sub(r'\s+', ' ', h) #replace bigger spaces with a single space
    return h

headline_list = [clean_headline(h) for h in headline_list]



No.of headlines in list:  53370
['Jim Cramer: A better way to invest in the Covid-19 vaccine gold rush', "Cramer's lightning round: I would own Teradyne", "Cramer's week ahead: Big week for earnings, even bigger week for vaccines", 'IQ Capital CEO Keith Bliss says tech and healthcare will rally', "Wall Street delivered the 'kind of pullback I've been waiting for,' Jim Cramer says"]


In [13]:
#adding start and end tags before each letter
word_dict = {}
for h in headline_list:
    chs = ['<S>'] + list(h) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        word_dict[bigram] = word_dict.get(bigram, 0) + 1 
        #returns 0 if none present, else corresponding dictionary bigram value

sorted(word_dict.items(), key = lambda kv : -kv[1])


[(('s', ' '), 97649),
 (('i', 'n'), 57628),
 (('e', ' '), 55114),
 (('e', 'r'), 46313),
 ((' ', 't'), 45456),
 (('n', ' '), 43812),
 (('e', 's'), 42438),
 ((' ', 's'), 41475),
 (('t', ' '), 41264),
 (('o', 'n'), 40240),
 (('r', 'e'), 36807),
 (('a', 'n'), 35004),
 ((' ', 'a'), 34113),
 (('r', ' '), 33112),
 (('o', 'r'), 32311),
 (('t', 'o'), 31404),
 (('s', 't'), 29151),
 ((' ', 'c'), 28865),
 (('a', 'r'), 28454),
 ((' ', 'o'), 27543),
 (('d', ' '), 26897),
 ((' ', 'f'), 26801),
 (('a', 'l'), 25530),
 (('o', ' '), 25435),
 ((' ', 'b'), 23926),
 ((' ', 'i'), 23390),
 (('t', 'e'), 23058),
 (('i', 't'), 22907),
 (('e', 'n'), 22518),
 (('n', 'g'), 22480),
 ((' ', 'p'), 21649),
 (('y', ' '), 21626),
 (('a', 't'), 20144),
 (('r', 'o'), 19930),
 (('d', 'e'), 19842),
 (('r', 'a'), 19798),
 (('v', 'e'), 19760),
 ((' ', 'r'), 19563),
 (('t', 'i'), 19243),
 (('l', ' '), 19098),
 (('t', 'h'), 19079),
 (('r', 'i'), 19013),
 (('s', 'e'), 18875),
 (('n', 'd'), 18767),
 (('e', 'a'), 18645),
 (('c', 'o

In [14]:
unique_words = set()

for headline in headline_list:
    for letter in list(headline):
        unique_words.add(letter)
    
unique_words = ['<S>', '<E>'] + sorted(unique_words)
print("Vocabulary size: ",len(unique_words))

Vocabulary size:  110


In [15]:
#number to word matrix
num2letter_dict = {i:letter for i, letter in enumerate(unique_words)}

#word to number matrix
letter2num_dict = {letter:i for i, letter in enumerate(unique_words)}


In [19]:
#creating the count matrix
N = torch.zeros((len(unique_words), len(unique_words)), dtype = torch.int32)

for h in headline_list:
    chs = ['<S>'] + list(h) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[letter2num_dict[ch1]][letter2num_dict[ch2]] += 1 #increment corresponding set of bigrams by 1
    

#now, calculating probabilities out of the counts
P = (N+1).float() / (N+1).float().sum(dim=1, keepdim=True)
print("Probability: ", P[letter2num_dict['s']])

Probability:  tensor([4.1920e-06, 6.4029e-02, 4.0935e-01, 5.8688e-05, 8.3840e-06, 4.1920e-06,
        4.1920e-06, 4.1920e-06, 4.1920e-06, 5.5460e-03, 4.1920e-06, 1.6768e-05,
        4.1920e-06, 8.3840e-06, 1.4261e-02, 1.6097e-03, 4.2759e-04, 7.5456e-05,
        4.1920e-06, 4.1920e-06, 4.1920e-06, 4.1920e-06, 8.3840e-06, 1.2576e-05,
        4.1920e-06, 4.1920e-06, 4.1920e-06, 4.1920e-06, 8.5601e-03, 1.2408e-03,
        1.3959e-03, 8.8032e-05, 4.1920e-06, 2.0960e-05, 4.1920e-06, 4.1920e-06,
        8.3840e-06, 4.1920e-06, 4.1920e-06, 4.1920e-06, 4.1920e-06, 4.1920e-06,
        4.1920e-06, 8.3840e-06, 4.1920e-06, 4.1920e-06, 1.6768e-05, 4.1920e-06,
        4.1920e-06, 4.1920e-06, 4.1920e-06, 8.3840e-06, 4.1920e-06, 8.3840e-06,
        8.3840e-06, 4.1920e-06, 8.3840e-06, 4.1920e-06, 4.0017e-02, 1.0480e-03,
        1.2358e-02, 1.7020e-03, 7.9128e-02, 7.5037e-04, 1.9283e-04, 3.9019e-02,
        5.3012e-02, 8.3840e-06, 7.5917e-03, 1.1859e-02, 4.9759e-03, 4.0956e-03,
        2.3169e-02, 1.6169

In [21]:
def generate_headline():
    current = '<S>'
    headline = []
    while True:
        get_index = letter2num_dict[current]
        prob_row = P[get_index] #row of probabilities of the respective character
        
        #now, considering prob_row as weights for multinomial
        char_index = torch.multinomial(prob_row, num_samples=1).item()
        current = num2letter_dict[char_index] #picks the char of the respective index

        #check for end char and break
        if current == '<E>':
            break
        headline.append(current)

    return ''.join(headline)

for _ in range(5):
    print(generate_headline())

#to intuitively think of torch.multinomial, think of a spinning wheel, where the higher probability gets a bigger chunk
#hence, it is like spinning that wheel, where it randomly selects the next occurence, the chances higher/lower based on the given probability of the character


S.Sts ceplank athat
USupscosen ain t b pus thaieestfufiekewoseentoracext Jinalarearure Arca hiay as Mameaing finousho vays ond alde Lyitist t
S. sup, tellede ari on courochoich stpilorines Cro d Gus pira thilaineanocoyicilden wn wen Dal macelexiris s stoone: ofrit o storend niver powis onife-lo Shos acte byary cof piding nte 190 —bue bewas dese owarang spro an s t Han
AX saberead ipereo wacthedeceds ck's ttonsatoom w cldeviecoirirs ariaftecusudselay ys ar Wanuimisesckede buang sirefrs fronn es msalemed 19, res bays ma'satimank ctendangh
U. m f Tir llererilindac o ve ono chile hinfe woon ipresin eanin In nddavetotill Trut and £19.Stikid t Ve ised es ck qushantor tonores imp aves: o e s
